In [2]:
import os
#load up required modules
import pandas as pd
from numpy.ma.core import reshape
from collections import defaultdict
from scipy.spatial import distance
from collections import defaultdict
import numpy as np

MAKING BINARY MATRIX, SKIP TO BLOCK 2 if you're starting with the provided eukaryotic binary matrix


In [18]:

##making binary matrix
###if you start with a proteome region file (not provided, download your fav version fom the Pfam FTP server; start with this
#Otherwise, to analyse already binary and filtered matrix start with the next cell
proteomeregion = pd.read_csv("Supplementary files/EPR /V35_proteome_regions.txt", sep="\t", header=None) ##REPLACE V35_PROTEOME_REGIONS_WITH THE VERSION YOU'RE WORKING WITH/downloaded from the ftp server
#The 1st column is a PFAM name
# The 2nd column is a Uniprot species ID
# The 3rd column is the number of times the column 1 PFAM occurs in the column 2 species
# The 4th column is the number of proteins in the column 2 species that contain the column 1
colnames= ['PfamID', 'SpeciesID', 'n_occur', 'n_prot']
proteomeregion.columns = colnames
##this ois an occurrance matrix - how many times does a Pfam occur in a species
matrix_n_occ = pd.pivot_table(proteomeregion, index="PfamID", columns="SpeciesID", values='n_occur', fill_value= 0)
#make binary
binary_matrix = (matrix_n_occ != 0).astype(int)

##filter the eukaryotes only using Supplementary File 6 - or one can filter species with less than x number of Pfams on the UNC

In [5]:
##load the binary eukaryotic matrix file name = eukaryoti matrix.csv or now - SUPPLEMENTARY FILE 1 untransposed_eukaryotic matrix_new.csv

##this matrix is a binary version of a file downloaded from Pfams ftp server - proteome_regions.txt
## It was also filtered to select only the eukaryotic species from Supplementary File 6 full eukspecies.xlsx
eukaryotic_binary_matrix = pd.read_csv('Supplementary files/SUPPLEMENTARY FILE 1 untransposed_eukaryotic matrix_new.csv', header=0, index_col=0)
eukaryotic_binary_matrix

,X1000413,X1001064,X1001832,X1001833,X1001938,X10020,X10029,X1003232,X10036,X100784,...,X9913,X9925,X993615,X9940,X994086,X994334,X9986,X99883,X9995,X999810
PF00001,0,0,0,0,0,1,1,0,1,1,...,1,1,0,1,0,0,1,1,1,0
PF00002,0,0,1,1,1,1,1,0,1,1,...,1,1,0,1,0,0,1,1,1,1
PF00003,0,0,0,0,0,1,1,0,1,1,...,1,1,0,1,0,1,1,1,1,0
PF00004,1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1
PF00005,1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
PF20158,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
PF20159,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
PF20160,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
PF20161,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [6]:
##load the Pfam description file
#file name: pfamDescriptions.csv
description = pd.read_csv("Supplementary files/EPR /pfamDescriptions.csv")
overlaps = description[description['PfamID'].isin(eukaryotic_binary_matrix.index)]
pfs_Description = overlaps[['PfamID', 'PfamName', 'PfamDescription']]

In [7]:
##load the trypanosome files and create a dictionary of Pfam Id as keys and protein and localisation as values
#file name: Trypanosomes_tryptag_modified.csv
trypspfamsgenes = pd.read_csv("Supplementary files/EPR /Trypanosomes_tryptag_modified.csv", sep=",", header=0)
trypspfamsgenes = trypspfamsgenes.iloc[:,1:]
trypspfamsgenes['Protein and Localisation'] = trypspfamsgenes['Gene ID'] + ' : ' + trypspfamsgenes['Localisation']

keys = trypspfamsgenes['Pfam ID'].astype(str).str.strip().str.strip('\'"')##some of the entries have a "'" and therefore aren't picked up, strip removes the '
values = trypspfamsgenes['Protein and Localisation']
trypspfamsgenes_dict = defaultdict(list)
for k, v in zip(keys, values):
    trypspfamsgenes_dict[k].append(v)

In [8]:
#testing
trypspfamsgenes_dict['PF12697']

['Tb927.11.14160 : ciliary plasm;cytoplasm;nuclear lumen',
 'Tb927.11.3690 : kinetoplast;mitochondrion',
 'Tb927.7.340 : mitochondrion',
 'Tb927.8.2110 : cytoplasm;mitochondrion;transport vesicle',
 'Tb927.8.5760 : endoplasmic reticulum;nuclear envelope']

In [9]:
##load the human genes and Pfam files - downloaded from ensembl - FULL human genes and pfam.xlsx
humangenesandpfams = pd.read_excel("Supplementary files/EPR /FULL human genes and pfam.xlsx")
keys = humangenesandpfams['Pfam ID']
values = humangenesandpfams['Gene name']
humanspfamsgenes_dict = defaultdict(set)
for k, v in zip(keys, values):
    humanspfamsgenes_dict[k].add(v)

In [10]:
humanspfamsgenes_dict['PF04712']

{'RSPH4A', 'RSPH6A'}

EPR analysis ON A MARKER PFAM

In [11]:
##this example enables users to input a single Pfam ID and obtain top X proximal Pfams to it based on EPR
def toppfamstable(pf, top_length):

    euclidean_distance = []
    for pfams in eukaryotic_binary_matrix.index:
        marker, target = eukaryotic_binary_matrix.loc[pf], eukaryotic_binary_matrix.loc[pfams]
        dist = distance.euclidean(marker,target)
        euclidean_distance.append((pfams,dist)) ##target Pfam and its distance to the marker

    df = pd.DataFrame(euclidean_distance, columns=['ClosestPfams', 'EuclideanDistance'])
    df_sorted = df.sort_values(by='EuclideanDistance', ascending=True) #Sort the list based on euclidean distance

    mapper_name = dict(zip(pfs_Description['PfamID'],pfs_Description['PfamName'])) ##obtain Pfam information
    mapper_description = dict(zip(pfs_Description['PfamID'],pfs_Description['PfamDescription']))
    df_sorted['Pfam_name'] = df_sorted['ClosestPfams'].map(mapper_name)
    df_sorted['Description'] = df_sorted['ClosestPfams'].map(mapper_description)
    df_sorted = df_sorted[0:top_length]#.reset_index(drop=True, inplace=True) ##so

    ##adding humans gene info and tryp protein and localisation
    df_sorted['Tryp genes'] = df_sorted['ClosestPfams'].map(trypspfamsgenes_dict).astype(str).str.strip('{}').str.strip('[]').str.replace("'", "", regex=False)
    df_sorted['Human gene'] = df_sorted['ClosestPfams'].map(humanspfamsgenes_dict).astype(str).str.strip('{}').str.replace("'", "", regex=False)

    return (df_sorted).reset_index(drop=True)
##to produce excel files hash out the above code and use this -> df_sorted.to_excel(f'sorted_pfams_{pf}.xlsx', index=False)

In [12]:
##change pf to any marker Pfam or a ciliary Pfam of your choice, change top_length paramater to change the number of closest Pfams that are displayed
table = toppfamstable(pf = 'PF03947',top_length=21)


In [13]:
table.to_clipboard(index=False)

In [14]:
table

,ClosestPfams,EuclideanDistance,Pfam_name,Description,Tryp genes,Human gene
0,PF03947,0.000000,Ribosomal_L2_C,"Ribosomal Proteins L2, C-terminal domain",Tb927.11.16280 : cytosolic large ribosomal sub...,"MRPL2, RPL8"
1,PF00297,5.196152,Ribosomal_L3,Ribosomal protein L3,Tb927.3.5610 : mitochondrial large ribosomal s...,"RPL3, RPL3L, MRPL3"
2,PF00514,5.291503,Arm,Armadillo/beta-catenin-like repeat,"Tb927.1.2670 : axoneme, Tb927.10.14320 : ciliu...","PKP2, ARMC6, ARMC8, KPNA3, PKP4, APC2, ARMC7, ..."
3,PF10609,5.291503,ParA,NUBPL iron-transfer P-loop NTPase,Tb927.10.1690 : ciliary plasm;cytoplasm;cytoso...,"NUBPL, NUBP1, NUBP2"
4,PF00025,5.385165,Arf,ADP-ribosylation factor family,Tb927.10.4250 : cytoplasm;microtubule cytoskel...,"ARL9, ARL4D, ARL16, ARL17A, ARF5, ARL4A, ARF3,..."
5,PF00006,5.385165,ATP-synt_ab,"ATP synthase alpha/beta family, nucleotide-bin...",Tb927.11.11690 : acidocalcisome;contractile va...,"ATP6V1B2, ATP5F1A, ATP5F1B, ATP6V1B1, ATP6V1A"
6,PF01248,5.385165,Ribosomal_L7Ae,Ribosomal protein L7Ae/L30e/S12e/Gadd45 family,Tb927.10.4110 : cytoplasm;cytosolic large ribo...,"RPL7A, GADD45A, GADD45G, GADD45B, SNU13, SECIS..."
7,PF17855,5.385165,MCM_lid,MCM AAA-lid domain,Tb927.10.10410 : MCM complex;cytoplasm;nucleop...,"MCM6, MCM4, MCMDC2, MCM3, MCM9, MCM7, nan, MCM..."
8,PF00181,5.477226,Ribosomal_L2,"Ribosomal Proteins L2, RNA binding domain",Tb927.11.16280 : cytosolic large ribosomal sub...,"MRPL2, RPL8"
9,PF02874,5.477226,ATP-synt_ab_N,"ATP synthase alpha/beta family, beta-barrel do...",Tb927.11.11690 : acidocalcisome;contractile va...,"ATP6V1B2, ATP5F1A, ATP5F1B, ATP6V1B1, ATP6V1A"


In [16]:
##this is to loop through the markers and save top X closest for each marker as excel files
##essentially the same was before but is useful if you'd like to look at closest Pfams to a number of marker ciliary Pfams
from scipy.spatial import distance
def toppfamstable(length, trypsdict, humansdict):
    ##get 100 closest from each of the marker pfams and write it as a csv
    MT_pfs = ['PF00119','PF00499','PF02790','PF00420','PF00507','PF00662','PF00510','PF00146','PF00033','PF00032','PF06455','PF00361','PF00115','PF00116','PF01059','PF00895','PF06444']
    ##pick a marker
    for pf in MT_pfs:
        euclidean_distance = []
        marker= eukaryotic_binary_matrix.loc[pf]
        ##
        for pfams in eukaryotic_binary_matrix.index:
                target = eukaryotic_binary_matrix.loc[pfams]
                dist = distance.euclidean(marker,target) ##scipy distance is SOMUCH FASTER
                euclidean_distance.append((pfams,dist))
        df = pd.DataFrame(euclidean_distance, columns=['ClosestPfams', 'EuclideanDistance'])
        df_sorted = df.sort_values(by='EuclideanDistance', ascending=True)

        mapper_name = dict(zip(pfs_Description['PfamID'],pfs_Description['PfamName']))
        mapper_description = dict(zip(pfs_Description['PfamID'],pfs_Description['PfamDescription']))
        df_sorted['Pfam_name'] = df_sorted['ClosestPfams'].map(mapper_name)
        df_sorted['Description'] = df_sorted['ClosestPfams'].map(mapper_description)
        df_sorted = df_sorted[0:length]#.reset_index(drop=True, inplace=True)

        ##adding humans and tryp localisation
        df_sorted['Tryp genes'] = df_sorted['ClosestPfams'].map(trypsdict).astype(str).str.strip('{}')
        df_sorted['Human gene'] = df_sorted['ClosestPfams'].map(humansdict).astype(str).str.strip('{}')

        df_sorted.to_excel(f'MTPFAMS_{pf}.xlsx', index=False)


In [17]:
toppfamstable(30, trypspfamsgenes_dict, humanspfamsgenes_dict)